## 0. Setup
Runtime → *Change runtime type* → **T4 GPU** (needed for training only; the submission check runs on CPU).

The cell below:
1. mounts Google Drive,
2. installs **exactly** the library versions Codabench uses (FLOP counts depend on the torch version),
3. copies the student package from Drive to the Colab disk (reading 15k images straight from Drive is slow),
4. links `data/checkpoints` back to Drive so checkpoints survive a disconnect.

Put the unzipped student package in `MyDrive/vqa_datathon/` first (or set `PKG_ZIP` to the zip on Drive).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive/vqa_datathon'   # unzipped student package lives here
PKG_ZIP = None                                    # or e.g. '/content/drive/MyDrive/vqa_datathon_student_package.zip'
WORK = '/content/vqa_datathon'                    # fast local copy

In [ ]:
%%capture pipout
!pip install torch==2.14.1 torchvision==0.29.1 transformers==5.20.2 pillow==12.3.0 numpy==2.2.6 tqdm==4.70.1 pyyaml==6.0.3

In [ ]:
# Colab imports some libraries (e.g. Pillow, numpy) at start-up. If the install above replaced one of them, the old
# copy is still in memory and imports fail (e.g. "cannot import name '_Ink' from 'PIL._typing'"). In that case the
# session restarts ONCE here: when it reconnects, choose Runtime -> Run all again (the install is then instant).
import sys
PINS = {'torch': '2.14.1', 'torchvision': '0.29.1', 'transformers': '5.20.2', 'PIL': '12.3.0', 'numpy': '2.2.6'}
stale = [m for m, v in PINS.items()
         if m in sys.modules and getattr(sys.modules[m], '__version__', v).split('+')[0] != v]
if stale:
    print(f'Pinned versions installed, but old {stale} already loaded: restarting the session once.')
    print('>>> When it reconnects, run Runtime -> Run all again. <<<', flush=True)
    import os, time; time.sleep(1); os.kill(os.getpid(), 9)
print('library versions OK')

In [ ]:
import os, subprocess, torch, torchvision, transformers
print('torch', torch.__version__, '| torchvision', torchvision.__version__, '| transformers', transformers.__version__,
      '| CUDA', torch.cuda.is_available())
assert torch.__version__.startswith('2.14.1'), 'Restart the runtime (Runtime -> Restart session) and rerun from the top'

if not os.path.exists(WORK):
    if PKG_ZIP:
        subprocess.run(['unzip', '-q', PKG_ZIP, '-d', '/content'], check=True)
        os.rename('/content/student_package', WORK)
    else:
        subprocess.run(['cp', '-r', DRIVE, WORK], check=True)
os.makedirs(f'{DRIVE}/checkpoints', exist_ok=True)
os.makedirs(f'{WORK}/data', exist_ok=True)
ck = f'{WORK}/data/checkpoints'
if not os.path.islink(ck):
    subprocess.run(['rm', '-rf', ck]); os.symlink(f'{DRIVE}/checkpoints', ck)
os.chdir(WORK)
print(open('README.md').read()[:600])

## 1. Point to your model
`MODEL_DIR` must contain `model.py` (with `build_model()`) and any helper files it reads.
`CKPT` is the `state_dict` file (`torch.save(model.state_dict(), path)`).

In [ ]:
MODEL_DIR = 'baseline/simple'                          # folder with model.py
CKPT = 'data/checkpoints/simple_baseline.pt'          # your checkpoint
OUT_ZIP = f'{DRIVE}/submission.zip'

## 2. Build and check the zip (CPU only, same program as Codabench)
This builds `submission.zip`, unzips it into a fresh folder, and runs **the same ingestion program Codabench runs**
on 2,000 val questions with **2 CPU threads, no GPU**. It prints the full error if something fails (Codabench shows
only a status code), plus val accuracy, params, FLOPs, the composite score and the estimated test runtime.

In [ ]:
!python eval/check_submission.py --model-dir {MODEL_DIR} --ckpt {CKPT} --out {OUT_ZIP} --n-val 2000 --threads 2

## 3. Upload
Download `submission.zip` from Drive and upload it on Codabench (*My Submissions*). Status 0 = ok. Choose which
submission appears on the leaderboard; that is the one ranked on the private test subset and run on demo day
(SHA256 printed above).